03 was a short smoke test. Here we do a proper baseline with logging via helpers/utils.py then try a few hyperparameter changes on val.

In [1]:
import sys
from pathlib import Path
from ultralytics import YOLO

In [2]:
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from helpers.utils import set_seed, yolo_log_kwargs
set_seed(42)

DATA_YAML = PROJECT_ROOT / "data" / "dataset.yaml"
WEIGHTS = PROJECT_ROOT / "models" / "yolo26n" / "yolo26n.pt"
RUNS = PROJECT_ROOT / "models" / "runs"

In [4]:
model = YOLO(str(WEIGHTS))

Same pretrained YOLO26n but longer training. Recall matters more than precision for early fire so we watch recall and mAP@0.5 on val.

In [5]:
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,
    close_mosaic=10,  # keep mosaic on until the last 10 epochs
    **yolo_log_kwargs(RUNS, "yolo26n_baseline", save_period=5),
)

New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.56  Python-3.11.15 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Arda\Desktop\Engineering\early-fire-detection\data\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

Best val (all): P=0.943, R=0.900, mAP50=0.944, mAP50-95=0.645.

Fire is a bit easier (R=0.916, mAP50=0.958) than smoke (R=0.883, mAP50=0.930)

In [8]:
from helpers.utils import read_yolo_metrics

run_dir = RUNS / "yolo26n_baseline"
df = read_yolo_metrics(run_dir)
print(df.tail(3))

print("best.pt metrics from train return:")
print(f"Precision: {results.box.mp}")
print(f"Recall:    {results.box.mr}")
print(f"mAP@0.5:   {results.box.map50}")
print(f"mAP@0.5:95:{results.box.map}")

    epoch     time  train/box_loss  train/cls_loss  train/dfl_loss  \
47     48  2597.86         0.97326         0.37377         0.00613   
48     49  2649.59         0.97961         0.38033         0.00614   
49     50  2697.49         0.97083         0.36929         0.00595   

    metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
47               0.94485            0.89749           0.94524   
48               0.91644            0.90545           0.94045   
49               0.92801            0.90365           0.94254   

    metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0  \
47              0.64625       1.14050       0.51490       0.01011  0.000116   
48              0.64366       1.14334       0.52252       0.01005  0.000083   
49              0.64383       1.14706       0.52285       0.01001  0.000050   

      lr/pg1    lr/pg2  
47  0.000116  0.000116  
48  0.000083  0.000083  
49  0.000050  0.000050  
best.pt metrics from train return:
Precis